# Homework 3: Machine Learning for Classification

## Machine Learning Zoomcamp 2026

**Topic:** Classification and Lead Scoring

### Objective
The objective of this homework is to explore a lead-scoring dataset and build an understanding of classification concepts, including exploratory data analysis, correlation, mutual information, feature selection, and logistic regression.

### Dataset
`course_lead_scoring_2026.csv`

### Learning Approach
This notebook documents the analysis, calculations, and reasoning used to answer each homework question. The goal is to understand the concepts and verify answers through code rather than rely on guesses.

**Author:** Shubham Kulkarni

In [14]:
#Loading data

In [15]:
import pandas as pd

df = pd.read_csv("course_lead_scoring_2026.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [16]:
# Check column names and data types
df.info()

# Count missing values in each column
print("\nMissing values per column:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB

Missing values per column:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
inter

In [17]:
# Question 1: Mode for industry

industry_mode = df["industry"].mode()

print("Mode of industry:", industry_mode.tolist())
print("\nIndustry frequency:")
print(df["industry"].value_counts())

Mode of industry: ['technology']

Industry frequency:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
Name: count, dtype: int64


### Interpretation — Question 1: Mode for Industry
### What I observed

When I checked the `industry` column, technology appeared more often than any other industry, with 1,173 entries. Retail came next with 925, followed by healthcare with 840.

So, the most common industry in this dataset is technology.

I found this useful because it gives me a quick idea of the kind of leads present in the dataset. However, having more leads from an industry doesn't automatically mean those leads are more likely to convert into customers. rate.

In [18]:
# Question 2: Compare correlations between the given pairs

features = [
    "interaction_count",
    "lead_score",
    "number_of_courses_viewed",
    "annual_income"
]

correlation_matrix = df[features].corr()

print(correlation_matrix.round(3))

                          interaction_count  lead_score  \
interaction_count                     1.000       0.929   
lead_score                            0.929       1.000   
number_of_courses_viewed              0.722       0.769   
annual_income                         0.106       0.231   

                          number_of_courses_viewed  annual_income  
interaction_count                            0.722          0.106  
lead_score                                   0.769          0.231  
number_of_courses_viewed                     1.000          0.138  
annual_income                                0.138          1.000  


### What I observed

I compared the correlation values for the pairs mentioned in the question. The strongest relationship was between `interaction_count` and `lead_score`, with a correlation of 0.929.

This suggests that leads with more interactions generally tend to have higher lead scores in this dataset. That makes sense as a pattern worth exploring, since interaction activity may be useful when evaluating leads.

However, correlation doesn't tell me that more interactions directly cause a higher lead score. It only shows that these two variables move together in this dataset.atasets.

In [19]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [20]:
%pip install --upgrade pip

In [21]:
# Question 3: Compare mutual information scores

from sklearn.metrics import mutual_info_score

features = [
    "industry",
    "location",
    "lead_source",
    "employment_status"
]

mi_scores = {}

for col in features:
    mi_scores[col] = mutual_info_score(
        df_train[col],
        y_train
    )

mi_scores = pd.Series(mi_scores).sort_values(ascending=False)

print(mi_scores.round(4))

lead_source          0.0343
employment_status    0.0190
industry             0.0020
location             0.0010
dtype: float64


### Interpreting the Mutual Information Scores

I compared the mutual information scores of four categorical features against `converted`, using only the training data. The highest score was for `lead_source`, at 0.0343, followed by `employment_status` at 0.0190.

This tells me that, among these four features, knowing where a lead came from provides the most information about its conversion outcome. However, the score is relatively small, so I wouldn't expect this feature alone to explain whether a lead will convert.

I also noticed that `industry` and `location` had much lower scores. In this dataset, they provide less information about conversion than `lead_source` and `employment_status` do.

## Data Preparation and Train-Validation-Test Split

Before training a classification model, I need to handle missing values and separate the data into training, validation, and test sets.

I'll replace missing categorical values with `NA` and missing numerical values with `0.0`, as required by the homework instructions. I'll also separate `converted`, which is the target I want the model to predict.

The training set will be used to train the model, the validation set to compare model performance, and the test set will be kept aside for final evaluation.

In [22]:
from sklearn.model_selection import train_test_split

# Make a copy so the original dataset stays unchanged
df_prepared = df.copy()

# Identify categorical and numerical columns
categorical_columns = df_prepared.select_dtypes(
    include="object"
).columns

numerical_columns = df_prepared.select_dtypes(
    include=["number"]
).columns.drop("converted")

# Fill missing values as required
df_prepared[categorical_columns] = (
    df_prepared[categorical_columns].fillna("NA")
)

df_prepared[numerical_columns] = (
    df_prepared[numerical_columns].fillna(0.0)
)

# Split the data using the exact homework settings
df_full_train, df_test = train_test_split(
    df_prepared, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Separate the target from the features
y_train = df_train["converted"]
y_val = df_val["converted"]
y_test = df_test["converted"]

df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])
df_test = df_test.drop(columns=["converted"])

print("Training set:", df_train.shape)
print("Validation set:", df_val.shape)
print("Test set:", df_test.shape)

print("\nMissing values in training features:",
      df_train.isnull().sum().sum())

Training set: (3000, 8)
Validation set: (1000, 8)
Test set: (1000, 8)

Missing values in training features: 0


## Question 4: Logistic Regression and Accuracy

So far, I've explored the dataset and compared the relationships between its features. Now I want to see how well a machine learning model can predict whether a lead converts.

I'll use logistic regression, which is commonly used for binary classification problems. Since the dataset contains categorical features such as `industry` and `lead_source`, I'll convert those categories into numerical features using one-hot encoding.

I'll train the model on the training data and evaluate it on the validation data. This will help me understand how well the model performs on data it wasn't trained on.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Identify categorical and numerical features
categorical_columns = df_train.select_dtypes(
    include="object"
).columns.tolist()

numerical_columns = df_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Encode categorical features and keep numerical features unchanged
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numerical",
            "passthrough",
            numerical_columns
        )
    ]
)

# Create the model with the required parameters
model = make_pipeline(
    preprocessor,
    LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )
)

# Train the model
model.fit(df_train, y_train)

# Predict conversions on the validation set
y_pred = model.predict(df_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", round(accuracy, 2))

Validation accuracy: 0.65


### What I observed

After training the logistic regression model, I got a validation accuracy of 0.65. This means the model correctly predicted the conversion outcome for 65% of the leads in the validation set.

This gives me a starting point for understanding how well the model can distinguish between leads that convert and those that don't. The result also reminds me that accuracy alone doesn't tell the whole story, especially if the two conversion classes are not equally represented.

I'd like to explore whether changing the features or model settings can improve the predictions in the next steps.

## Question 5: Feature Selection

In this question, I want to find out whether every feature contributes equally to the model's predictions. I'll train the model with all features first, then remove `lead_source`, `number_of_courses_viewed`, and `interaction_count` one at a time.

I'll compare each model's validation accuracy with the original model. The feature whose removal causes the smallest change in accuracy will be the answer.

This is a simple way to understand whether a feature adds much value to the model.

In [24]:
# Question 5: Compare accuracy after removing each feature

features_to_remove = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

# Original model accuracy, using the unrounded value
original_accuracy = accuracy_score(y_val, model.predict(df_val))

results = {}

for feature in features_to_remove:
    # Remove one feature from training and validation data
    X_train_reduced = df_train.drop(columns=[feature])
    X_val_reduced = df_val.drop(columns=[feature])

    # Build a fresh model with the same settings
    categorical_reduced = X_train_reduced.select_dtypes(
        include="object"
    ).columns.tolist()

    numerical_reduced = X_train_reduced.select_dtypes(
        include=["number"]
    ).columns.tolist()

    preprocessor_reduced = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_reduced
            ),
            ("numerical", "passthrough", numerical_reduced)
        ]
    )

    reduced_model = make_pipeline(
        preprocessor_reduced,
        LogisticRegression(
            solver="liblinear",
            C=1.0,
            max_iter=1000,
            random_state=42
        )
    )

    reduced_model.fit(X_train_reduced, y_train)
    reduced_accuracy = accuracy_score(
        y_val,
        reduced_model.predict(X_val_reduced)
    )

    results[feature] = {
        "accuracy_without_feature": reduced_accuracy,
        "difference": abs(original_accuracy - reduced_accuracy)
    }

print("Original accuracy:", original_accuracy)

for feature, result in results.items():
    print(
        f"{feature}: "
        f"accuracy={result['accuracy_without_feature']:.6f}, "
        f"difference={result['difference']:.6f}"
    )

Original accuracy: 0.645
lead_source: accuracy=0.642000, difference=0.003000
number_of_courses_viewed: accuracy=0.643000, difference=0.002000
interaction_count: accuracy=0.601000, difference=0.044000


### What I observed

I removed each of the three features separately and compared the resulting validation accuracy with the original model's accuracy of 0.645.

When I removed `lead_source`, the accuracy dropped to 0.642. Removing `number_of_courses_viewed` gave an accuracy of 0.643, while removing `interaction_count` reduced it to 0.601.

The smallest change came from removing `number_of_courses_viewed`. Based on this experiment, it seems to contribute the least to the model's predictions among these three features.

This doesn't mean that the number of courses viewed is useless in every situation. It only tells me that, with this dataset and model setup, removing it had the smallest effect on validation accuracy.

## Question 6: Tuning the Regularization Parameter

In this final question, I'll experiment with different values of `C` in logistic regression. The value of `C` controls how strongly the model is regularized, which affects how complex a model it can learn.

I'll train four models using the same training and validation sets, changing only `C`. Then I'll compare their validation accuracies to find which value performs best.

This helps me understand how a model's settings can affect its predictions, even when the dataset and features remain the same.

In [25]:
# Question 6: Compare different values of C

C_values = [0.000001, 0.00001, 0.0001, 0.001]

accuracy_results = {}

for C_value in C_values:
    # Use the same preprocessing as Question 4
    model_tuned = make_pipeline(
        preprocessor,
        LogisticRegression(
            solver="liblinear",
            C=C_value,
            max_iter=1000,
            random_state=42
        )
    )

    # Train and evaluate the model
    model_tuned.fit(df_train, y_train)
    y_pred_tuned = model_tuned.predict(df_val)

    accuracy_tuned = accuracy_score(y_val, y_pred_tuned)
    accuracy_results[C_value] = accuracy_tuned

    print(
        f"C = {C_value}: "
        f"accuracy = {accuracy_tuned:.3f}"
    )

C = 1e-06: accuracy = 0.598
C = 1e-05: accuracy = 0.598
C = 0.0001: accuracy = 0.613
C = 0.001: accuracy = 0.645


### What I observed

I tested four values of `C` while keeping the training data, validation data, and other model settings the same. The two smallest values gave an accuracy of 0.598. Increasing `C` to 0.0001 improved it to 0.613, and the best result came from `C = 0.001`, with an accuracy of 0.645.

This helped me see that the regularization setting can make a noticeable difference to a model's performance. In this experiment, the stronger regularization used by the smaller values of `C` did not perform as well as `C = 0.001`.

The best value here is based on validation accuracy for this dataset and these model settings. It doesn't mean that the same value will always work best for other datasets.

# Concepts, Decisions, and the Reasoning Behind Them

## 1. Why do we explore the data before building a model?

A machine learning model learns patterns from the data we give it. If I don't understand the dataset first, I might use the wrong columns, overlook missing values, or misunderstand what the model is supposed to predict.

That is why I started by checking the number of rows and columns, the data types, and the missing values. These checks helped me understand what information was available and what needed attention before modelling.

For example, the `converted` column had no missing values. This was important because it is the target variable the model needs to learn to predict. If the target values were missing, those records could not directly provide the correct outcomes needed for supervised training.

**The main idea:** Data exploration helps me make informed decisions before training a model, rather than discovering problems after the model has been built.

## 2. Why do we need to handle missing values?

Many machine learning algorithms cannot work directly with missing values. Even when a library supports them, missing data can still affect the reliability of the analysis.

In this homework, I replaced missing categorical values with `"NA"` and missing numerical values with `0.0`, following the assignment instructions.

These replacements have different meanings. `"NA"` gives a categorical column an explicit category for records where the value was missing. Replacing a number with `0.0` allows the numerical column to be processed without a missing value, but it does not mean the original value was actually zero.

This distinction matters. For example, an unknown annual income is not necessarily an income of zero. In a real project, I would investigate what missingness means and compare suitable imputation strategies rather than assume that one method is best for every dataset.

**The main idea:** Missing-value handling makes the data usable, but the method chosen can affect what the model learns.

## 3. Why do we separate features from the target?

The target, `converted`, is the outcome we want to predict. The remaining columns provide information that the model can use to make that prediction.

Keeping the target separate from the features makes this distinction explicit. It also helps prevent accidental target leakage, where information about the answer is included in the inputs in a way that would not be available when making real predictions.

For this assignment, `converted` contains the two possible outcomes: `1` for a conversion and `0` for no conversion.

**The main idea:** The model should learn from information available before the outcome is known, not from the answer itself.

## 4. Why do we split the data into training, validation, and test sets?

A model can perform well on the data it has already seen without performing equally well on new data. We need a way to check whether the patterns it learns generalise beyond its training examples.

That is why the dataset was split into three parts:

- **Training set — 3,000 rows:** Used to fit the model.
- **Validation set — 1,000 rows:** Used to compare model choices, features, and parameter settings.
- **Test set — 1,000 rows:** Reserved for the final evaluation after those choices have been made.

The validation set helps me decide which approach to use. The test set gives me a more independent estimate of how the selected model performs on unseen data.

If I kept changing the model based on test results, I would gradually tune my decisions to that test set too. Keeping it separate helps avoid that problem.

**The main idea:** We need separate data for learning, decision-making, and final evaluation so that performance estimates are more trustworthy.

## 5. Why do we use different techniques to study relationships between variables?

Correlation and mutual information both help me investigate relationships, but they do not measure the same thing.

### Correlation

I used Pearson correlation to compare numerical variables. It measures the strength and direction of a linear relationship.

The correlation between `interaction_count` and `lead_score` was approximately `0.929`, indicating a strong positive linear association in this dataset.

Why is that useful? It helps identify numerical variables that move together and gives me clues about patterns worth investigating.

However, correlation does not establish causation, and Pearson correlation may miss relationships that are not linear.

### Mutual information

I used mutual information to compare categorical features with the `converted` target. It measures how much knowing one variable reduces uncertainty about another.

Among the four features tested, `lead_source` had the highest score on the training set, at approximately `0.0343`.

Why use this instead of correlation? Mutual information can capture statistical dependence that a linear correlation measure may not capture, and it can be used with categorical variables.

The scores should be compared within this experiment. A larger score indicates more information about the target, but it does not automatically mean that a feature will improve the model's validation accuracy.

**The main idea:** Different measures answer different questions. I should choose a technique based on the type of variables and the relationship I want to investigate.

## 6. Why do we use one-hot encoding?

The dataset contains categorical values such as `industry`, `location`, and `lead_source`. Logistic regression requires numerical input, so these categories need to be represented numerically.

One-hot encoding creates a separate binary indicator for each category. This allows the model to learn different relationships for different categories without treating category labels as numbers with an artificial order.

For example, assigning `technology = 1`, `retail = 2`, and `healthcare = 3` could incorrectly suggest that healthcare is greater than retail or technology. One-hot encoding avoids introducing that arbitrary numerical ordering.

In this notebook, I used `OneHotEncoder(handle_unknown="ignore")` so that an unfamiliar category encountered during prediction would not cause the transformation to fail.

**The main idea:** Encoding converts categorical information into a form the model can use without inventing an order between categories.

## 7. Why did we choose logistic regression?

The target has two outcomes, so this is a binary classification problem. Logistic regression is a useful starting point because it estimates the probability of the positive class and uses that probability to produce a class prediction.

It is also relatively straightforward to train and interpret compared with many more complex models.

The model does not simply calculate the mean of the target. It learns how the input features relate to the probability of conversion.

**The main idea:** Logistic regression gives me a practical baseline against which I can compare more complex classification methods later.

## 8. Why do we measure accuracy?

Accuracy is the proportion of predictions that match the actual outcomes:

Accuracy = Number of correct predictions / Total number of predictions

I used validation accuracy to understand how often the model correctly classified the validation records. The original model achieved approximately `0.645`, or 64.5%.

Why is that useful? It gives me a simple, consistent measure for comparing the models in this homework.

But accuracy has a limitation. If one outcome is much more common than the other, a model may achieve seemingly good accuracy by favouring the majority class. Accuracy also treats all mistakes equally, even when some mistakes might be more costly in a real business problem.

That is why I would also examine the confusion matrix, precision, recall, and F1-score before deciding whether the model is useful.

**The main idea:** Accuracy is a useful starting point, not a complete assessment of a classification model.

## 9. Why do we remove features one at a time?

Feature selection helps me investigate whether each input contributes useful predictive information.

I removed `lead_source`, `number_of_courses_viewed`, and `interaction_count` separately, trained a new model each time, and compared validation accuracy with the original model.

Removing `number_of_courses_viewed` produced the smallest change in accuracy in this experiment. Removing `interaction_count` produced a much larger drop.

Why do this experiment? A feature may appear relevant during exploration but add little to the model's actual predictions once the other features are included. Testing feature removal helps reveal that difference.

However, a small change in accuracy does not prove that a feature is universally unimportant. The result depends on the dataset, the model, the evaluation metric, and the other features available.

**The main idea:** Feature selection should be supported by model evaluation, not based only on intuition or correlation.

## 10. Why do we tune the regularization parameter?

Logistic regression includes regularization to discourage overly complex model coefficients. Regularization can help reduce overfitting, where a model learns patterns specific to its training data that do not generalise well.

In scikit-learn, `C` is the inverse of regularization strength:
- Smaller `C` means stronger regularization.
- Larger `C` means weaker regularization.

I compared four values of `C` while keeping the data split and other model settings unchanged. Of those values, `C = 0.001` achieved the highest validation accuracy.

Why keep the other settings unchanged? If I changed several things at once, I would not know which change caused the performance difference.

The best value is only the best among the values tested. A broader search or a different dataset could produce a different result.

**The main idea:** Hyperparameter tuning is a controlled experiment. I change a setting, measure its effect, and use validation results to make a decision.

## 11. How do these steps fit together?

The workflow is connected:

1. **Explore the data** to understand the columns, target, and missing values.
2. **Prepare the data** so the features can be used consistently.
3. **Split the data** to separate model fitting from evaluation.
4. **Investigate relationships** to understand which features may contain useful information.
5. **Encode categorical variables** so the model can process them.
6. **Train a baseline model** to establish a reference point.
7. **Evaluate its predictions** using validation data.
8. **Test feature removal and parameter changes** to see whether performance improves or stays similar.
9. **Evaluate the final selected model on the test set** only after the modelling decisions are complete.

Each step answers a different question. Skipping one can make it harder to understand the data, compare models fairly, or trust the final performance estimate.



# Final Notes: What I Learned from Classification

## 1. What was the objective of this homework?

In this homework, I worked with a lead-scoring dataset containing 5,000 records. Each record represents a lead, with information such as its source, industry, employment status, annual income, number of courses viewed, interaction count, and lead score.

The goal was to explore the data and understand how machine learning can predict whether a lead converts into a customer.

The target variable was `converted`:
- `1` means the lead converted.
- `0` means the lead did not convert.

This is a **binary classification problem** because the model predicts one of two possible outcomes.

Rather than jumping straight into model training, I first explored the dataset, studied relationships between features, prepared the data, and then evaluated a logistic regression model.

## 2. Understanding the dataset before modelling

I started by loading the CSV using Pandas and checking its shape, column names, data types, and missing values.

The dataset contained 5,000 rows and 9 columns. I found missing values in several columns, including `industry`, `location`, `annual_income`, and `lead_score`. The target column, `converted`, had no missing values.

I checked this because missing data can affect calculations and model training. It is important to understand the data before deciding how to prepare it.

For this homework, I replaced missing categorical values with `"NA"` and missing numerical values with `0.0`, following the assignment instructions.

I also kept the original dataframe unchanged by creating a copy for preparation. This made it easier to revisit the raw data if I needed to check something.

## 3. What I learned from each question

### Question 1: Finding the mode

**What I did:** I used Pandas' `mode()` function to find the most frequent category in the `industry` column.

**Why I did it:** The mode is useful for summarising categorical data. Unlike the mean or median, it identifies the category that occurs most often.

**What I found:** Technology was the most common industry, appearing 1,173 times among the non-missing values.

**What I learned:** A dataset can contain more records from one category than another. However, the most common category is not necessarily the one with the highest conversion rate. Frequency and conversion performance are two different things.

### Question 2: Finding the strongest correlation

**What I did:** I calculated the Pearson correlation coefficients for the numerical variable pairs provided in the question.

**Why I did it:** Correlation helps me understand how two numerical variables move in relation to each other. A positive correlation means that higher values of one variable tend to occur with higher values of the other.

**What I found:** The strongest correlation among the given options was between `interaction_count` and `lead_score`, at approximately `0.929`.

**What I learned:** These two variables have a strong positive linear relationship in this dataset. This is an association, not proof that increasing interactions directly causes a higher lead score.

I also learned that correlation is useful for exploring relationships, but it does not tell me whether a feature will necessarily improve a model's predictions.

### Question 3: Comparing mutual information

**What I did:** I calculated mutual information between four categorical features and the target, `converted`, using the training data.

**Why I did it:** Mutual information measures how much information one variable provides about another. It can identify useful relationships between categorical features and a classification target that correlation may not describe well.

**What I found:** `lead_source` had the highest mutual information score at approximately `0.0343`, followed by `employment_status` at `0.0190`.

**What I learned:** Among the four features tested, knowing where a lead came from provided the most information about its conversion outcome. However, the score was relatively small, so I should not expect this feature alone to predict conversion accurately.

I also learned that feature relevance depends on the relationship being measured. A feature can have a strong relationship with another numerical feature without necessarily being the most informative feature for predicting the target.

### Question 4: Training a logistic regression model

**What I did:** I prepared the data, separated the target from the input features, encoded the categorical columns using one-hot encoding, and trained a logistic regression model.

**Why I did it:** Logistic regression is a common starting point for binary classification. It estimates the probability of an outcome and predicts one of the available classes.

One-hot encoding was necessary because the model requires numerical input. It represents categorical values as separate binary features instead of assigning arbitrary numerical values to categories.

I split the data into training, validation, and test sets:
- Training: 3,000 rows
- Validation: 1,000 rows
- Test: 1,000 rows

The training set was used to fit the model. The validation set was used to evaluate and compare model choices. The test set was kept separate for final evaluation.

**What I found:** The validation accuracy was approximately `0.645`, displayed as `0.65` when rounded to two decimal places.

**What I learned:** Accuracy measures the proportion of predictions that were correct. In this case, the model correctly classified approximately 64.5% of the validation examples.

However, accuracy does not tell me everything about model performance. I would also want to examine precision, recall, and the confusion matrix, particularly if correctly identifying converted leads is more important than other types of predictions.

### Question 5: Understanding feature selection

**What I did:** I removed `lead_source`, `number_of_courses_viewed`, and `interaction_count` one at a time. Each time, I trained a fresh model and compared its validation accuracy with the original model.

**Why I did it:** Feature selection helps me understand whether every input variable contributes useful information to the model. Removing a feature that adds little value can simplify a model without significantly reducing its performance.

**What I found:**

| Feature removed | Validation accuracy |
|---|---:|
| `lead_source` | `0.642` |
| `number_of_courses_viewed` | `0.643` |
| `interaction_count` | `0.601` |

The original model's accuracy was `0.645`. Removing `number_of_courses_viewed` produced the smallest change, reducing accuracy by only `0.002`.

**What I learned:** In this experiment, `number_of_courses_viewed` contributed the least to validation accuracy among the three features tested. Removing `interaction_count`, on the other hand, caused a much larger reduction.

This does not mean that the number of courses viewed is always useless. Its importance can change with the dataset, the model, and the other features available.

### Question 6: Tuning the regularization parameter

**What I did:** I trained logistic regression models using four values of `C`: `0.000001`, `0.00001`, `0.0001`, and `0.001`.

**Why I did it:** Regularization helps control model complexity and can reduce overfitting. In scikit-learn's logistic regression, `C` is the inverse of regularization strength. Smaller values of `C` mean stronger regularization, while larger values mean weaker regularization.

I changed only `C` and kept the training and validation sets and the other model settings the same. This allowed me to compare the results fairly.

**What I found:**

| Value of `C` | Validation accuracy |
|---:|---:|
| `0.000001` | `0.598` |
| `0.00001` | `0.598` |
| `0.0001` | `0.613` |
| `0.001` | `0.645` |

The best validation accuracy among the tested values was `0.645`, achieved with `C = 0.001`.

**What I learned:** Model settings can make a noticeable difference to performance. In this experiment, the stronger regularization associated with the smaller values of `C` did not perform as well as `C = 0.001`.

This is the best value only among the values tested. It does not guarantee that `0.001` will be best for every dataset or every classification problem.

## 4. Why the train, validation, and test split matters

One important lesson from this homework is that I should not evaluate a model using only the data it has already seen.

The three datasets have different purposes:

- **Training set:** Used to learn the model parameters.
- **Validation set:** Used to compare models, features, and hyperparameters.
- **Test set:** Reserved for a final evaluation after the modelling choices have been made.

If I repeatedly make decisions based on test-set performance, I risk indirectly tuning the model to that test set. Keeping it separate gives me a more reliable estimate of how the final model performs on unseen data.

## 5. My main takeaways

This homework helped me connect several classification concepts rather than treating them as separate definitions.

- Exploratory data analysis helps me understand the dataset before modelling.
- Correlation measures linear association between numerical variables.
- Mutual information helps compare how informative features are about a target.
- One-hot encoding makes categorical data suitable for a model that expects numerical inputs.
- Logistic regression provides a starting point for binary classification.
- Validation accuracy helps compare model choices, but it is not the only measure of performance.
- Feature selection tests whether removing a variable changes the model's performance.
- Regularization controls model complexity, and hyperparameter tuning helps identify suitable settings.

Most importantly, I learned to change one thing at a time when comparing models. This makes it easier to understand why the results change and to reproduce the experiment later.

## 6. What I would explore next

If I continued working on this problem, I would:

1. Examine the distribution of `converted` to see whether the classes are balanced.
2. Build a confusion matrix and calculate precision, recall, and F1-score.
3. Investigate whether missing-value handling affects the results.
4. Compare logistic regression with another classification model.
5. Evaluate the selected model on the untouched test set once the modelling decisions are finalised.

These steps would help me understand not just how accurate the model is, but also what kinds of mistakes it makes and whether it is useful for the actual lead-scoring problem.

## Final reflection

This assignment gave me practice with the complete early-stage classification workflow: inspecting data, preparing features, exploring relationships, training a model, evaluating predictions, and comparing model settings.

I now have a better understanding of why each step matters and how to use the results to make the next decision. I also learned that a model's score is only one part of the analysis; understanding the data, the assumptions, and the limitations is equally important.

SHUBHAM ANANT KULKARNI